In [0]:
                                        # AWS GLUE & AWS Crawlers
AWS Glue: is a pipeline in AWS, can perform AETL operations. Low-No code solution
DNS->Spark->Warehouse->Report
E       T       L

Glue gas 2 components:
1) Data Catalog: You register the tables. You set/have data in s3 and you use (Glue/specially Crawler) to build the tables/catalog. Crawler will automatically fetch the medatada of all the files/files inside the folders and can register them in the Catalog automatically.
                Flow: AWS s3 -> Crawler -> Catalog tables
followup Q) You have created data in s3 why do you need to register in table?
= Supose you have csv/parquet files in s3. Now for Big Data Analysis/ query the data/analyze the data. For those particular things you need to register those tables in Catalog.       

2) Visual ETl: This can perform entire ELT pipeline using Low No Code. 
            Flow: User -> Glue -> workflow
------------
Practical AWS Glue: Search AWS GLUE in search bar in portal. You will find it is empty and will tell to create Database. But it is not a database and it is a component/layer you use and you will create objects in Glue. 
            Add database -> Name: demodb -> Create datbase

nodes: is activity. For ETL: Extract the data it is node, Transform the data it is node, Load the data it is node. 
        +node: Amazon s3 (name: Amazon s3 source, s3 source type: s3 location RAW, data format: csv,save, IAM role: you want me to access the s3 but where is the permission: AWS IAM: GlueAccessS3New->start session->Data preview: view data in Live mode) -> + Node: for Transformations: Select Fields(select all the columns as per requirements->save)-> +Node: Target: AWS Glue Data Catalog (select parent node, create a flder name "processed" in s3 and give this dentination to this target system otherwise it will not create folder, format: Delta Lake, Enable Inferschema: otherwise error(Because it pass the schema to subsequent steps, so select all fields)) ->save & run

Script: Whatever drag and drop you have done it is writing in code/script
Amazon s3 vs AWS Glue Data Catalog:
Amazon s3: if you pick it for target it will dumps the data into the folder.
AWS Glue Data Catalog: It will dumps the data as well register the data in the data catalog as well.  

Now Lets Do a Practical Example2:
            User->dump data in s3 as csv file-> Now we need to trigger lambda Function ->ETL
+------+      +--------+      +----------+      +-------------------------------------------+
|      |      |        |      |          |      |             ETL Process                   |
| USER |----->| AWS S3 |----->|  LAMBDA  |----->|  +--------+      +-------+      +-------+  |
|      | CSV  | (Drop) |      | (Trigger)|      |  | AWS S3 |----->| SPARK |----->| WARE- |  |
+------+      +--------+      +----------+      |  | (Raw)  |      |       |      | HOUSE |  |
                                                |  +--------+      +-------+      +-------+  |
                                                |      E              T              L      |
                                                +-------------------------------------------+

lambda: have code to trigger our pipeline -> Search lambda in search bar +create lambda (Name: trigger lambda, Runtime:Python3.12, Execution Role: use an existing role: lambdaAccessS3)->IAM(Roles: click lambadAccessS3, add permission role: AWSGlueConsoleFullAccess, AWSs3FullAccess, AWSlambdaExecute)->create-> (Now you need to write the code to build the logic for it/client but no need to remember the code which is good) -> search: AWS python sdk: search GLUE: start job run-> (Go to lambda once) instead of s3 put Glue
import json
import boto3
def lambda_handler(event,context):
    glue_client = boto3.client('glue')
    glue_client.start_job_run(
        JobName='demopipeline'
    )
    return{
        'statusCode': 200,
        'body': json.dumps('Pipeline Triggered')
    }
-> Deploy-> Test


In [0]:
                                        # AWS Lambda Functions
It is like Azure Logic Apps but not like that because it supports workflow . so It i s a serverless functions for AWS.

                    in short :    # This is Entire Story of AWS Lambda:
AWS Lambda: You can literally manage your code without managing the servers. 
Lambda is a serverless service which will manage the server, it can autoscale up-and-down (will not wastage of resources for all time running server, because EC2 running all time. you can ryn python code all demand using Lambda. And only charge you for the time_frame/moment it is running) 

Means: you want to run custom code/python/java/scala code. You know if you want to run any functions, compile any code or any interpreter of any code you need to install interpreter/compiler on your machine. Then you run the code in machine. You do not need to run the resources it will simply manage everything for you and provision the backend.

EC2: is vm  

-- Here you do not need to upgrade any kind of OS/compiler/interpreters. You do not need to install anything. Just click & submit the python code. Not saying you will totally replace ec2 because there are Many complex task as well can be done through EC2.

               				            - - ->   [ EC2 ]
  +-------+                           /           
  |       |                          /
  | Python|                         /          - Wastage of resources
  |  Code | -----------------------<           - Scalability of resources
  +-------+                         \
                                     \
                                      \ - ->   [ Lambda ]     ✅ 

                                               - Serverless Compiler/Interpreter
                                               - Can easily scale up/down
                                               - Can be triggered on demand

Use case: Web Application, Machine Learning, Data Processing.
# ------------
#Practical:
AWS search bar->functions->Create Function: (Function Name: lambdaansh, Runtime: Python 3.12, Change default execution role: Create a new role with bsic Lambda permissions)->create & you will see vs code

# to add roles:
AWS search bar->IAM->Roles->lambda-ansh-roles(Which is previously set)->add permessions
#vs code:
import json
def lambda_handler(event, context):
    
    return {
        'statusCode': 200,
        'body': json.dumps('Hello from Lambda!')
    }
# Understand vs code:
# event: it is nothing but a just input coming from the previous activity.
# context: It is metadata not important. AI Generates: is the context of the function which is running    
# lambda_handler: function name 
Now test: as Test previous event not there create event name: test_event_new

Monitor->view cloud watch logs. 
For any kind of change in vs code aws->deploy->Test

Many more are there I have not write it due to time constant watch here : https://youtu.be/Ae3xyW2Ef0c


In [0]:
                                            # AWS Athena
Serverless SQL pool. https://aws.amazon.com/athena/

The file is unstructured you can query in Athena and Athena will do the top of the data.
If you open and run query select * from raw?
= Because you created raw table using crawler and crawler automatically registerted this data in database.

Glue and Athena both are connected to same data catalog